# 🧬 HAM10000 Skin Lesion Model Training
### Health Disease Predictor - External Training on Google Colab

**Before running this notebook:**
1. Go to `Runtime` → `Change runtime type` → Select **GPU (T4)** → Save
2. Upload your `HAM10000` dataset to Google Drive (see Step 1 below)
3. Run all cells in order

**What this notebook does:**
- Trains EfficientNet-B0 on 7 skin disease classes
- Uses GPU for fast training (~20 mins vs hours on CPU)
- Saves `best_model.pth` and `classes.json` to Google Drive
- You then copy those 2 files back to your project

## ✅ Step 0: Check GPU is available

In [ ]:
import torch
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))
    print('GPU Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️ WARNING: No GPU! Go to Runtime → Change runtime type → GPU')

## 📁 Step 1: Mount Google Drive

**Before running this cell, upload your dataset to Google Drive:**
- Create a folder in your Google Drive called `HAM10000`
- Upload these files/folders inside it:
  - `HAM10000_metadata.csv`
  - `HAM10000_images_part_1/` (folder with images)
  - `HAM10000_images_part_2/` (folder with images)

> **Dataset source:** Download from Kaggle: https://www.kaggle.com/datasets/kmader/skin-lesion-analysis-toward-melanoma-detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

## 📂 Step 2: Set Paths

In [ ]:
import os

# ======================================================
# CHANGE THIS if your dataset is in a different folder
DATA_DIR = '/content/drive/MyDrive/HAM10000'
OUTPUT_DIR = '/content/drive/MyDrive/HAM10000/trained_model'
# ======================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check dataset exists
metadata_path = os.path.join(DATA_DIR, 'HAM10000_metadata.csv')
if os.path.exists(metadata_path):
    print('✅ Metadata found!')
else:
    print(f'❌ Metadata NOT found at: {metadata_path}')
    print('Please upload HAM10000_metadata.csv to your Google Drive HAM10000 folder')

part1 = os.path.join(DATA_DIR, 'HAM10000_images_part_1')
part2 = os.path.join(DATA_DIR, 'HAM10000_images_part_2')
print(f'Part 1 exists: {os.path.exists(part1)}')
print(f'Part 2 exists: {os.path.exists(part2)}')

## 📦 Step 3: Install Dependencies

In [ ]:
!pip install -q torch torchvision tqdm scikit-learn matplotlib pandas Pillow
print('✅ All packages installed!')

## 🧠 Step 4: Training Code

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

torch.manual_seed(42)
print('✅ Libraries loaded!')


# ── Dataset ──────────────────────────────────────────
class HAM10000Dataset(Dataset):
    def __init__(self, df, image_dirs, transform=None):
        self.df = df
        self.image_dirs = image_dirs
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row['image_id']
        label = row['label']
        image_path = None
        for img_dir in self.image_dirs:
            p = os.path.join(img_dir, f'{image_id}.jpg')
            if os.path.exists(p):
                image_path = p
                break
        if image_path is None:
            raise FileNotFoundError(f'Image {image_id}.jpg not found')
        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


# ── Transforms ───────────────────────────────────────
def get_transforms(augment=True):
    if augment:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])


# ── Data Loaders ─────────────────────────────────────
def create_dataloaders(data_dir, batch_size=64, val_split=0.2):
    metadata_path = os.path.join(data_dir, 'HAM10000_metadata.csv')
    df = pd.read_csv(metadata_path)
    print(f'Loaded {len(df)} images')
    classes = sorted(df['dx'].unique().tolist())
    print(f'Classes ({len(classes)}): {classes}')
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    df['label'] = df['dx'].map(class_to_idx)
    print('\nClass distribution:')
    for cls in classes:
        print(f'  {cls}: {len(df[df["dx"]==cls])} images')
    train_df, val_df = train_test_split(df, test_size=val_split, random_state=42, stratify=df['label'])
    print(f'\nTrain: {len(train_df)} | Val: {len(val_df)}')
    image_dirs = [
        os.path.join(data_dir, 'HAM10000_images_part_1'),
        os.path.join(data_dir, 'HAM10000_images_part_2'),
    ]
    train_ds = HAM10000Dataset(train_df, image_dirs, get_transforms(True))
    val_ds   = HAM10000Dataset(val_df,   image_dirs, get_transforms(False))
    class_counts = train_df['label'].value_counts().sort_index().values
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[l] for l in train_df['label']]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,   num_workers=2, pin_memory=True)
    return train_loader, val_loader, classes


# ── Model ─────────────────────────────────────────────
def create_model(num_classes):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, num_classes)
    return model


# ── Training Loop ─────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, labels in tqdm(loader, desc='Training'):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(inputs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, pred = out.max(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Validating'):
            inputs, labels = inputs.to(device), labels.to(device)
            out = model(inputs)
            loss = criterion(out, labels)
            total_loss += loss.item() * inputs.size(0)
            _, pred = out.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100. * correct / total


print('✅ All functions defined. Ready to train!')

## 🚀 Step 5: Start Training
This will train for **20 epochs** on GPU. Should take ~20-30 minutes.

In [ ]:
# ── Hyperparameters (you can change these) ────────────
EPOCHS = 20         # More epochs = better accuracy
BATCH_SIZE = 64     # Reduce to 32 if you get out-of-memory errors
LEARNING_RATE = 1e-3
# ─────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# Load data
print('\n📂 Loading data...')
train_loader, val_loader, classes = create_dataloaders(DATA_DIR, batch_size=BATCH_SIZE)

# Create model
print('\n🧠 Creating model...')
model = create_model(len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training loop
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'\n🚀 Starting training for {EPOCHS} epochs...\n')

for epoch in range(EPOCHS):
    print(f'\n══════════ Epoch {epoch+1}/{EPOCHS} ══════════')

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Train  →  Loss: {train_loss:.4f}  |  Acc: {train_acc:.2f}%')
    print(f'Val    →  Loss: {val_loss:.4f}  |  Acc: {val_acc:.2f}%')

    scheduler.step(val_loss)

    if val_acc > best_acc:
        best_acc = val_acc
        out_dir = Path(OUTPUT_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), out_dir / 'best_model.pth')
        with open(out_dir / 'classes.json', 'w') as f:
            json.dump({'classes': classes}, f, indent=2)
        print(f'  ✅ New best! Saved model (val_acc={val_acc:.2f}%)')

print(f'\n🏆 Training complete! Best Val Accuracy: {best_acc:.2f}%')
print(f'📁 Model saved to: {OUTPUT_DIR}')

## 📊 Step 6: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss', color='#e74c3c')
ax1.plot(history['val_loss'],   label='Val Loss',   color='#3498db')
ax1.set_title('Loss Curve', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc', color='#e74c3c')
ax2.plot(history['val_acc'],   label='Val Acc',   color='#3498db')
ax2.set_title('Accuracy Curve', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_metrics.png'), dpi=150)
plt.show()
print('📊 Plot saved!')

## ⬇️ Step 7: Download the Trained Model

Run this to download `best_model.pth` and `classes.json` directly to your computer.
Then copy them to your project at:
- `ml_image/models/image_model/best_model.pth`
- `ml_image/models/image_model/classes.json`

In [ ]:
from google.colab import files

print('⬇️ Downloading model files...')
files.download(os.path.join(OUTPUT_DIR, 'best_model.pth'))
files.download(os.path.join(OUTPUT_DIR, 'classes.json'))
print('✅ Done! Now copy these 2 files to:')
print('   ml_image/models/image_model/best_model.pth')
print('   ml_image/models/image_model/classes.json')

## 🔁 Step 8 (Optional): Test the Model Before Downloading

In [ ]:
# Quick sanity check on validation set
model_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
classes_path = os.path.join(OUTPUT_DIR, 'classes.json')

with open(classes_path) as f:
    loaded_classes = json.load(f)['classes']

test_model = create_model(len(loaded_classes)).to(device)
test_model.load_state_dict(torch.load(model_path, map_location=device))

print(f'✅ Model loaded successfully!')
print(f'📋 Classes: {loaded_classes}')

final_loss, final_acc = validate(test_model, val_loader, criterion, device)
print(f'\n🏆 Final Validation Accuracy: {final_acc:.2f}%')
print(f'📉 Final Validation Loss: {final_loss:.4f}')